# Retail Sales Performance & Profit Margin Analytics
**Author:** Aaryan Tulsankar ([@TulsankarAaryan](https://github.com/TulsankarAaryan))

This notebook demonstrates an end-to-end analytical pipeline using Python and an in-memory SQLite database to compute sub-category profit margins, classify operational tiers with SQL `CASE` statements, and generate profitability-weighted visualizations.

## 1. Setup & In-Memory Database Initialization

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Connect to in-memory SQLite database
conn = sqlite3.connect(':memory:')

# Transaction dataset
data = {
    'Order_ID': [1, 2, 3, 4, 5, 6],
    'Category': ['Tech', 'Tech', 'Furniture', 'Furniture', 'Office', 'Office'],
    'Sub_Category': ['Phones', 'Laptops', 'Chairs', 'Tables', 'Paper', 'Pens'],
    'Sales': [1000, 2000, 1500, 1200, 500, 300],
    'Profit': [100, 250, -50, -100, 10, 5]
}

sales_df = pd.DataFrame(data)
sales_df.to_sql('Sales_Data', conn, index=False, if_exists='replace')
print("SQLite In-Memory Database successfully initialized!")
sales_df

## 2. SQL Sub-Category Margin Aggregation

In [ ]:
query_margin = """
SELECT
    Category,
    Sub_Category,
    SUM(Sales) AS Total_Sales,
    SUM(Profit) AS Total_Profit,
    ROUND((CAST(SUM(Profit) AS FLOAT) / SUM(Sales)) * 100, 2) AS Margin_Percentage
FROM Sales_Data
GROUP BY Category, Sub_Category
ORDER BY Margin_Percentage DESC;
"""

margin_analysis = pd.read_sql(query_margin, conn)
margin_analysis

## 3. Visualization: Sales by Profitability Heatmap

In [ ]:
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

ax = sns.barplot(
    data=margin_analysis,
    x='Sub_Category',
    y='Total_Sales',
    palette='RdYlGn',
    hue='Sub_Category',
    legend=False
)

plt.title('Sales Revenue by Sub-Category (Color = Profitability Trend)', fontsize=14, weight='bold', pad=15)
plt.ylabel('Total Sales ($)', fontsize=12)
plt.xlabel('Sub-Category', fontsize=12)

# Annotate profit margins
for idx, row in margin_analysis.iterrows():
    val = row['Margin_Percentage']
    ax.text(idx, row['Total_Sales'] + 30, f"{val:+.1f}%", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Advanced SQL: Dynamic Performance Tier Classification

In [ ]:
query_tiering = """
SELECT
    Sub_Category,
    Total_Sales,
    Total_Profit,
    Margin_Percentage,
    CASE
        WHEN Margin_Percentage > 10 THEN 'High Performer'
        WHEN Margin_Percentage BETWEEN 0 AND 10 THEN 'Stable'
        ELSE 'Review Required'
    END AS Performance_Status
FROM (
    SELECT
        Sub_Category,
        SUM(Sales) AS Total_Sales,
        SUM(Profit) AS Total_Profit,
        ROUND((CAST(SUM(Profit) AS FLOAT) / SUM(Sales)) * 100, 2) AS Margin_Percentage
    FROM Sales_Data
    GROUP BY Sub_Category
)
ORDER BY Total_Sales DESC;
"""

tier_report = pd.read_sql(query_tiering, conn)
tier_report